# 04 — NNP Stability Scoring (MACE)

This notebook estimates a **ΔΔG-like stability proxy** for each candidate mutation using **MACE**, a neural network potential (NNP), instead of a classical force field (Amber/CHARMM) or full DFT. The idea: relax the local geometry around each mutation site with MACE and compare the resulting **atomization energy** (E0-baseline-corrected — see the sanity-check section below for why raw total energy isn't safe to compare directly) against the same local region in the wild type. A mutation whose relaxed local energy comes out *lower* (more favorable) than the wild type's is a candidate for genuinely improved local stability, not just "evolutionarily tolerated" (notebook 02) or "still folds confidently" (notebook 03) — this is the first *physics-based* signal in the pipeline.

**Why only a local pocket, not the whole 327-residue chain:** relaxing thousands of atoms with an NNP, iteratively, is expensive — and most of a mutation's structural consequence is local anyway. We freeze everything except the mutated residue and its immediate neighbors (residues with a CA within `POCKET_CUTOFF` of the mutated residue's CA), and only let that local pocket relax. This is standard practice in computational ΔΔG estimation, not a shortcut unique to this notebook.

**Inputs:** the WT and top-8 candidate structures already folded by ESMFold in notebook 03 (`results/structures/*.pdb`) and the combined table from that notebook (`results/structure_scores.csv`).

**Run this in Google Colab** (`Runtime` → `Change runtime type`; GPU strongly recommended this time — an earlier CPU-only run took ~10+ minutes per candidate; with GPU it should be much faster). Written but **not executed locally** — `mace-torch` pulls in PyTorch, which this project keeps out of the lightweight local `.venv` by design (same call as notebook 02). The grafting step (see below) *was* verified locally, since it only needs Biopython + numpy.

### Running in Colab: mount Google Drive

Same setup as notebook 02: upload the project (`data/`, `notebooks/`, `results/`) to `MyDrive/enzyme-design-ai/` on Google Drive, then open this notebook via Drive's right-click → "Open with Google Colaboratory" (the in-Colab file picker has been unreliable for freshly-uploaded files). Edit `PROJECT_DIR` below if your folder differs.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/enzyme-design-ai"  # edit if your folder differs
    os.chdir(f"{PROJECT_DIR}/notebooks")
    print(f"Working directory: {os.getcwd()}")
else:
    print("Not running in Colab — skipping Drive mount, using local relative paths.")

## Setup

In [ ]:
%pip install -q mace-torch ase biopython py3Dmol

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from ase.io import read as ase_read
from ase.optimize import LBFGS
from ase.constraints import FixAtoms
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
STRUCTURES_DIR = RESULTS_DIR / "structures"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load notebook 03's outputs

The candidate table (with `esm_score`, `mean_plddt`, `at_subunit_interface`, `near_terminus`) and the already-folded WT + variant structures — no re-folding needed here.

In [ ]:
structure_scores = pd.read_csv(RESULTS_DIR / "structure_scores.csv")
print(f"{len(structure_scores)} candidates loaded from notebook 03")
structure_scores[["mutation", "position", "esm_score", "at_subunit_interface", "near_terminus"]]

## Graft each mutation onto the WT scaffold

**Why this step exists:** `WT.pdb` and each `{mutation}.pdb` from notebook 03 are *independently* predicted ESMFold structures, not the same backbone with one residue swapped. An earlier version of this notebook compared their total potential energy directly and got energy deltas in the thousands of eV — physically implausible for a single point mutation. The cause: two separate folding runs differ slightly everywhere (not just at the mutation site), and that noise, sitting in the atoms we "freeze," swamped the real local signal.

**Fix:** graft each mutation onto a single, shared reference structure (`WT.pdb`) instead of using two independently-folded structures. For each candidate: take the mutated residue's side-chain conformation from its own ESMFold prediction (a real, chemically sensible geometry), then use the **Kabsch algorithm** (optimal rigid-body superposition — the same method behind PyMOL's `align`) to superimpose that residue's backbone (N, CA, C, O) onto the same residue's backbone in `WT.pdb`, and apply that same rotation+translation to the side-chain atoms before splicing them into a copy of `WT.pdb`. Every other atom in the result is then identical to `WT.pdb` — the comparison now isolates the mutation's local effect instead of mixing in noise from two different global folds.

Verified locally before relying on it (pure Biopython + numpy, no GPU needed): for all 8 candidates, every backbone atom outside the mutated residue matches `WT.pdb` to within 1e-8 A, and each grafted residue has the correct name and heavy-atom count for its amino acid.

In [ ]:
import numpy as np


def kabsch_fit(mobile: np.ndarray, target: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (R, t) such that mobile @ R.T + t optimally superimposes onto target."""
    mobile_center = mobile.mean(axis=0)
    target_center = target.mean(axis=0)
    mobile_c = mobile - mobile_center
    target_c = target - target_center

    H = mobile_c.T @ target_c
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T

    t = target_center - R @ mobile_center
    return R, t


def apply_transform(R: np.ndarray, t: np.ndarray, points: np.ndarray) -> np.ndarray:
    return points @ R.T + t

In [ ]:
import copy
from Bio.PDB import PDBIO, Atom

bio_parser = PDBParser(QUIET=True)
BACKBONE_ATOMS = {"N", "CA", "C", "O"}


def graft_mutation_onto_wt(wt_path: Path, variant_path: Path, position: int, out_path: Path) -> float:
    wt_structure = bio_parser.get_structure("wt", wt_path)
    variant_structure = bio_parser.get_structure("variant", variant_path)

    wt_residue = next(r for r in next(wt_structure[0].get_chains()) if r.id[1] == position)
    variant_residue = next(r for r in next(variant_structure[0].get_chains()) if r.id[1] == position)

    backbone_names = ["N", "CA", "C", "O"]
    wt_backbone = np.array([wt_residue[name].coord for name in backbone_names])
    variant_backbone = np.array([variant_residue[name].coord for name in backbone_names])
    R, t = kabsch_fit(variant_backbone, wt_backbone)
    backbone_rmsd = np.sqrt(np.mean(np.sum((apply_transform(R, t, variant_backbone) - wt_backbone) ** 2, axis=1)))

    new_structure = copy.deepcopy(wt_structure)
    target_residue = next(r for r in next(new_structure[0].get_chains()) if r.id[1] == position)

    for atom_name in [a.get_name() for a in list(target_residue)]:
        if atom_name not in BACKBONE_ATOMS:
            target_residue.detach_child(atom_name)

    for atom in variant_residue:
        if atom.get_name() in BACKBONE_ATOMS:
            continue
        new_coord = apply_transform(R, t, atom.coord.reshape(1, 3))[0]
        new_atom = Atom.Atom(
            atom.get_name(), new_coord, atom.get_bfactor(), atom.get_occupancy(),
            atom.get_altloc(), atom.get_fullname(), atom.get_serial_number(), atom.element,
        )
        target_residue.add(new_atom)

    target_residue.resname = variant_residue.resname

    io = PDBIO()
    io.set_structure(new_structure)
    io.save(str(out_path))
    return backbone_rmsd

In [ ]:
STRUCTURES_GRAFTED_DIR = RESULTS_DIR / "structures_grafted"
STRUCTURES_GRAFTED_DIR.mkdir(parents=True, exist_ok=True)

wt_path = STRUCTURES_DIR / "WT.pdb"

for row in structure_scores.itertuples():
    variant_path = STRUCTURES_DIR / f"{row.mutation}.pdb"
    out_path = STRUCTURES_GRAFTED_DIR / f"{row.mutation}.pdb"
    rmsd = graft_mutation_onto_wt(wt_path, variant_path, row.position, out_path)
    print(f"{row.mutation}: backbone RMSD before alignment = {rmsd:.4f} A -> grafted onto WT scaffold")

print("All candidates grafted onto the WT scaffold.")

## Define the local pocket

For a given structure and a mutation position, find every residue whose CA atom is within `POCKET_CUTOFF` of the mutated residue's CA — that's the pocket allowed to relax. Everything else gets frozen via ASE's `FixAtoms` constraint.

We build the freeze mask with Biopython (residue-level distance check) and apply it to the same PDB file read by ASE — both read the file in the same atom order, so a per-atom "is this atom in a pocket residue?" boolean lines up directly between the two libraries.

In [ ]:
POCKET_CUTOFF = 8.0  # Angstrom, CA-CA distance

bio_parser = PDBParser(QUIET=True)


def pocket_freeze_indices(pdb_path: Path, position: int, cutoff: float = POCKET_CUTOFF) -> list[int]:
    structure = bio_parser.get_structure(pdb_path.stem, pdb_path)
    chain = next(structure[0].get_chains())
    residues = [r for r in chain if is_aa(r, standard=True)]

    center = next(r for r in residues if r.id[1] == position)["CA"]
    pocket_res_ids = {
        r.id[1] for r in residues
        if (r["CA"] - center) <= cutoff
    }

    frozen = []
    atom_index = 0
    for r in residues:
        in_pocket = r.id[1] in pocket_res_ids
        for _atom in r:
            if not in_pocket:
                frozen.append(atom_index)
            atom_index += 1
    return frozen

## Load MACE and define the relaxation

`mace_off` loads a pretrained MACE-OFF23 foundation model — trained broadly on organic molecules (H, C, N, O, F, P, S, Cl), not protein-specific, but a reasonable and fast off-the-shelf NNP for this kind of exploratory local relaxation, consistent with this project's goal of substituting a classical force field / DFT with a neural network potential.

The relaxation freezes everything outside the pocket, then runs a bounded local optimization (`LBFGS`, `fmax=0.05 eV/A`, capped at 100 steps) and returns the final potential energy.

In [ ]:
from mace.calculators import mace_off

calc = mace_off(model="small", device=device)
print("MACE-OFF23 (small) loaded")

### Sanity check: element-composition artifact in the raw total energy

A first full run of this notebook (before this cell existed) produced `nnp_energy_delta_eV` values in the thousands of eV, and — the giveaway — the magnitude correlated almost perfectly with how much a mutation changes *element composition*, not with genuine local (in)stability: swaps involving sulfur (`L1M` adds S, `C105S` removes S) gave the largest deltas (-9802, +8788 eV), while same-element carbon-only changes (`A266V/I`, `A228V`, all Ala→bigger-hydrocarbon) gave much smaller ones (-2065 to -3111 eV).

The cause: MACE's raw total energy is referenced to per-element "isolated atom" baseline energies (E0s), which differ by up to thousands of eV between elements (e.g. sulfur vs. carbon). Comparing raw total energies between a WT pocket and a mutated pocket with a *different* element composition mixes this baseline difference into the result — nothing to do with packing or stability.

MACE separates this out internally: after a calculation, `calc.results["node_energy"]` holds **per-atom energies with the isolated-atom E0 already subtracted** (`calc.results["energies"]` would be the un-subtracted version). Summing `node_energy` gives the atomization energy — comparable across different element compositions, since the baseline is removed for every atom, not just the ones in common.

Confirm this API exists before relying on it in the full run below (untested locally — no GPU/MACE here):

In [ ]:
_test_atoms = ase_read(STRUCTURES_DIR / "WT.pdb")
_test_atoms.calc = calc
_test_atoms.get_potential_energy()

print("calc.results keys:", list(_test_atoms.calc.results.keys()))
assert "node_energy" in _test_atoms.calc.results, (
    "Expected 'node_energy' in MACE calculator results - check the installed mace-torch version's API "
    "(this notebook was written against the documented behavior, not tested locally)."
)
node_energy = _test_atoms.calc.results["node_energy"]
print("node_energy shape:", getattr(node_energy, "shape", None), " n_atoms:", len(_test_atoms))
print("sum(node_energy) [atomization energy, eV]:", float(node_energy.sum()))
print("get_potential_energy() [raw total energy, eV]:", _test_atoms.get_potential_energy())

In [ ]:
def relax_pocket_energy(pdb_path: Path, position: int) -> tuple[float, float, int]:
    atoms = ase_read(pdb_path)
    frozen = pocket_freeze_indices(pdb_path, position)
    atoms.set_constraint(FixAtoms(indices=frozen))
    atoms.calc = calc

    opt = LBFGS(atoms, logfile=None)
    opt.run(fmax=0.05, steps=100)

    total_energy = atoms.get_potential_energy()
    atomization_energy = float(atoms.calc.results["node_energy"].sum())

    n_free = len(atoms) - len(frozen)
    return total_energy, atomization_energy, n_free

## Relax WT and each variant's pocket, compute the ΔΔG proxy

For each candidate: relax the wild-type structure's pocket centered at that same position, relax the *grafted* variant structure's pocket, and take the energy difference — using the **atomization energy** (E0-corrected, see the sanity-check section above), not the raw total energy, since the raw total energy is not comparable when a mutation changes element composition. A **negative** `nnp_atomization_delta_eV` means the variant's local pocket relaxed to a *lower* (more favorable) atomization energy than the wild type's — a proxy signal for improved local stability at that site. The raw totals are still recorded for reference/debugging, but `nnp_atomization_delta_eV` is the column to trust.

One remaining, inherent asymmetry: the mutated side chain itself has a different atom count than the wild type's, so the two pockets still aren't perfectly atom-for-atom identical. Atomization energy fixes the *element-type* baseline mismatch, not this atom-*count* asymmetry — that's a property of comparing different amino acids this way, not something either grafting or E0-correction can fully remove. The same issue affects established ΔΔG tools like Rosetta.

In [ ]:
wt_path = STRUCTURES_DIR / "WT.pdb"

rows = []
for row in structure_scores.itertuples():
    print(f"Relaxing pocket for {row.mutation} (position {row.position})...")

    wt_total, wt_atomization, wt_n_free = relax_pocket_energy(wt_path, row.position)
    variant_path = STRUCTURES_GRAFTED_DIR / f"{row.mutation}.pdb"
    variant_total, variant_atomization, variant_n_free = relax_pocket_energy(variant_path, row.position)

    rows.append({
        "mutation": row.mutation,
        "position": row.position,
        "wt_pocket_total_energy_eV": wt_total,
        "variant_pocket_total_energy_eV": variant_total,
        "wt_pocket_atomization_energy_eV": wt_atomization,
        "variant_pocket_atomization_energy_eV": variant_atomization,
        "nnp_atomization_delta_eV": variant_atomization - wt_atomization,
        "pocket_n_free_atoms_wt": wt_n_free,
        "pocket_n_free_atoms_variant": variant_n_free,
    })
    print(f"  atomization delta = {rows[-1]['nnp_atomization_delta_eV']:+.3f} eV")

nnp_df = pd.DataFrame(rows)
nnp_df

## Combine into the final ranking table

In [ ]:
final_df = structure_scores.merge(nnp_df, on=["mutation", "position"])
final_df = final_df.sort_values("nnp_atomization_delta_eV").reset_index(drop=True)

out_path = RESULTS_DIR / "variant_rankings.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
final_df

## Written analysis (fill in after running)

Once real numbers come back from Colab, work through these questions together rather than trusting any single column in isolation:

- Which candidates have a **favorable `nnp_atomization_delta_eV`** (negative) *and* a strong `esm_score` *and* are **not** flagged `at_subunit_interface` or `near_terminus`? Those are the strongest combined candidates.
- Where do the signals **disagree** — e.g. a great `esm_score` but a poor (positive) `nnp_atomization_delta_eV`, or vice versa? Disagreement is informative, not just noise: ESM-2 reflects evolutionary tolerance across many homologs, MACE reflects a physical, local energy estimate from one specific relaxed geometry — they can legitimately disagree, and rank differently, especially for surface-exposed positions where evolution tolerates more than raw local energetics might suggest, or vice versa for buried positions.
- Do any of the interface-flagged candidates (`Y250N/S/P`) still look attractive despite the flag? Remember `at_subunit_interface` is a caution flag from a real geometric fact (notebook 03), not a verdict — but this notebook still only relaxed each mutation's *monomer* pocket, not the true inter-subunit contact (that would need mutating directly into the notebook-01 tetramer coordinates — not implemented here, a reasonable next step if any interface-flagged candidate otherwise looks strong).
- Sanity-check magnitude one more time: `nnp_atomization_delta_eV` values should now be much smaller than the old raw-total-energy deltas (thousands of eV) — expect roughly single-digit to low-tens of eV for a well-behaved local mutation effect. If they're still in the hundreds/thousands, something is still off and worth digging into before trusting the ranking.

## Visualize the top candidates (py3Dmol)

Two views, using the winning position (266) and the rejected interface-adjacent one (250) for contrast:

1. The whole WT tetramer (from notebook 01) with position 266 highlighted on every chain (it's a homotetramer — a real construct carries this mutation on all four copies at once), the catalytic residues (Thr12/Thr89) marked for reference, and position 250 marked to visually confirm why it was flagged (sitting right at the A-C subunit interface).
2. A close-up overlay of WT, A266I, and A266V at position 266, to see the two winning side chains next to the original alanine.

`py3Dmol` is lightweight (no GPU/PyTorch) — unlike the MACE relaxation above, this section runs the same locally or in Colab. It renders an interactive widget inline; nothing to verify numerically here, just look at the rendered structure once this runs in a live notebook.

In [ ]:
import py3Dmol

tetramer_pdb = (DATA_DIR / "pdb" / "pdb3eca.ent").read_text()

view = py3Dmol.view(width=800, height=600)
view.addModel(tetramer_pdb, "pdb")
view.setStyle({}, {"cartoon": {"colorscheme": "chain"}})

# winning position, all 4 chains (a real construct mutates every copy at once)
view.addStyle({"resi": "266"}, {"stick": {"color": "orange"}})
# catalytic residues, chain A only, for reference
view.addStyle({"chain": "A", "resi": ["12", "89"]}, {"stick": {"color": "red"}})
# rejected interface-adjacent position, chain A only
view.addStyle({"chain": "A", "resi": "250"}, {"stick": {"color": "magenta"}})

view.zoomTo()
view.show()

In [ ]:
close_up = py3Dmol.view(width=800, height=600)

variants = [
    ("WT", STRUCTURES_DIR / "WT.pdb", "gray"),
    ("A266I", STRUCTURES_GRAFTED_DIR / "A266I.pdb", "orange"),
    ("A266V", STRUCTURES_GRAFTED_DIR / "A266V.pdb", "cyan"),
]
for label, path, color in variants:
    close_up.addModel(Path(path).read_text(), "pdb")
    close_up.setStyle({"model": -1}, {"cartoon": {"color": color, "opacity": 0.25}})
    close_up.addStyle({"model": -1, "resi": "266"}, {"stick": {"color": color}})

close_up.zoomTo({"resi": "266"})
close_up.show()

## Summary

- Loaded the WT and top-8 variant structures already folded in notebook 03 — no re-folding needed.
- **Grafted** each mutation onto the WT scaffold (Kabsch superposition) instead of comparing two independently-folded ESMFold structures directly — fixed one source of noise (differing global folds), verified locally.
- **Discovered a second, larger issue** after the first real Colab run with grafting: `nnp_energy_delta_eV` (raw total energy) was still in the thousands of eV, correlating with how much a mutation changes *element composition* (e.g. sulfur swaps in `L1M`/`C105S` gave the largest deltas), not with genuine stability. Cause: MACE's raw total energy includes per-element isolated-atom reference energies (E0s), which differ by thousands of eV between elements and don't cancel out when comparing pockets with different atom composition. **Fixed** by using MACE's `node_energy` output (E0-already-subtracted, per the MACE docs) summed into an atomization energy, and comparing `nnp_atomization_delta_eV` instead.
- For each candidate, relaxed only the local pocket (mutated residue + neighbors within 8 A) with a pretrained MACE-OFF23 potential, freezing everything else, and compared the relaxed atomization energy against the same pocket relaxed in the wild type.
- Combined `esm_score` (02), `mean_plddt` / `at_subunit_interface` / `near_terminus` (03), and `nnp_atomization_delta_eV` (04) into `results/variant_rankings.csv` — the final ranking table for this pipeline.
- **Confirmed working end-to-end (2026-08-07):** `nnp_atomization_delta_eV` came out in the range -19.2 to +32.9 eV — the sane, small magnitude expected. **A266I** (-19.2 eV) and **A266V** (-15.9 eV) are the top picks: the only two candidates where ESM-2 score, structure/pLDDT checks, and this physics-based signal all agree favorably. `L1M`'s top ESM-2 score is confirmed as a likely terminus artifact (near-neutral energy, +1.6 eV). `Y250N/S/P` are the weakest candidates (worst energies, +26 to +33 eV, *and* interface-flagged).
- Visualized the winning position (266, all four chains) alongside the catalytic residues and the rejected interface position (250) on the WT tetramer, plus a close-up overlay of WT/A266I/A266V side chains, with `py3Dmol`.
- **Caveats carried forward:** MACE-OFF23 is a general organic-chemistry potential, not protein-specific. The relaxation is monomer-only and pocket-only — it does not capture whether an interface-flagged mutation still permits correct tetramer assembly (a possible follow-up: apply the same Kabsch-grafting technique to the notebook-01 tetramer instead of the monomer). The remaining atom-*count* asymmetry between different amino acids' side chains is not fully resolved by either grafting or the E0 correction.
- **This is the last notebook in the core pipeline.** What's left: writing up the README with this final analysis for GitHub publication.